# Lab 27: AI Agents and Workflow Automation

**Duration:** 4 hours. **Level:** intermediate. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Tool calling (Lab 25) answers one question. An **AI agent** works toward a **goal** that needs several steps: it decides which tool to use, looks at the result, decides the next step, and stops when the goal is done. In this lab you build a small, **bounded** agent for the simulated building. It has a step limit, a record of what it did, error handling, and a rule that a **person must approve** every action that changes the real world. You run the **same agent** with tools that are Python functions (Lab 25) and with tools from an MCP server (Lab 26). This is the milestone of the language labs: **an AI agent with direct tools and MCP integration**.

**By the end of this lab you can:**
1. Explain what an agent is: a model in a loop with tools, a goal, a state and a stop rule.
2. Build an agent loop with a step limit, a repeat guard, error handling and a log.
3. Add an approval gate so that the agent cannot change the real world without a person.
4. Run the same agent with direct tools and with MCP tools, and show both workflows.
5. Test the agent with goals, check the real result in Python, and name where agents fail.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Safety.** An agent acts without asking at every step. That is its strength and its danger. Give it only the tools it needs, limit the steps, record everything, and ask a person before anything that changes the real world. The building here is simulated.

## 2. Background

**Agent = model + tools + loop + state + stop rule.**

| Part | What it is | In this lab |
|---|---|---|
| Goal | What the user wants done, in a sentence | `run_agent(goal)` |
| Model | Decides the next step | Gemma 4 E2B through `labview.llm_message` |
| Tools | What the agent can read and do | Direct functions (Lab 25) or an MCP server (Lab 26) |
| Loop | Think, act, observe, think again | `run_agent` |
| State | What happened so far | The conversation and the `log` |
| Stop rule | When to end | The model answers with text, or the **step limit** |

**Think, act, observe.** The model thinks which tool to call (think), your program runs it (act), the result goes back to the model (observe). This repeats until the model decides that the goal is done and writes a report.

**Workflow automation.** A workflow is a fixed task done again and again, for example *every morning check all rooms and switch on a fan where it is too warm*. An agent can do a workflow whose steps are not fixed: how many rooms, which ones are warm, what to do. A plain script would need an `if` for every case.

**Bounded.** A small model can loop, repeat itself or wander. Four rules keep an agent bounded:

| Rule | Why |
|---|---|
| **Step limit** (`max_steps`) | The agent cannot run forever |
| **Repeat guard** | The same call with the same arguments three times is a loop. Stop it. |
| **Errors as results** | A failing tool returns text, the model can try something else |
| **Approval gate** | A person says yes before a tool that changes the real world |

**Do not trust the report.** The agent writes its report from what it believes. It can be wrong (it can say it switched something off when it did not). After a run, check the **real state** with your own Python code, and read the log.

**The agent does not need MCP.** MCP is one way to offer tools. The agent only needs a list of tools and a way to call them. In this lab a *provider* has two methods, `tools()` and `call(name, arguments)`. One provider uses the functions directly, the other uses an MCP session. The agent code is the same for both.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(27)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (20 min)

### Step 1: The Server, the Building and the MCP Library
You need the language model server (Lab 20), the simulated building (Lab 25) and the MCP library (Lab 26). The cell checks them. If the library is missing it installs it.

In [ ]:
import asyncio
import csv
import inspect
import json
import os
import subprocess
import sys
import time
from datetime import datetime

import labview
import sensor_tools as building

try:
    import mcp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp>=2,<3"], check=True)
    import mcp

print("Language model server ready:", labview.llm_ready())
print("Tools of the building:", list(building.FUNCTIONS))
print("Tools that change the real world (need approval):", sorted(building.CONSEQUENTIAL))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which tools only read, and which change the building:
- Why do set_fan and open_door need approval, and set_light is allowed without it (your opinion):

---
# Part B: The Agent Loop (50 min)

### Step 1: The Providers
A **provider** gives the agent its tools. Both providers have the same two methods, so the agent does not care where the tools come from. `DirectProvider` calls the Python functions of `building`. `McpProvider` uses an MCP session (you use it in Part F).

In [ ]:
class DirectProvider:
    """Tools that are Python functions in this notebook (Lab 25)."""
    name = "direct"

    def __init__(self):
        self.schemas = building.all_schemas()

    async def tools(self):
        return self.schemas

    async def call(self, name, arguments):
        try:
            return json.dumps(building.FUNCTIONS[name](**arguments))
        except Exception as error:                   # unknown tool, wrong arguments, unknown room ...
            return f"ERROR: {error}"


class McpProvider:
    """Tools that come from an MCP server (Lab 26)."""
    name = "mcp"

    def __init__(self, session):
        self.session = session

    async def tools(self):
        listed = await self.session.list_tools()
        return [{"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.input_schema}} for t in listed.tools]

    async def call(self, name, arguments):
        result = await self.session.call_tool(name, arguments)
        return ("ERROR: " if result.is_error else "") + (result.content[0].text if result.content else "")


print("Direct tools:", [t["function"]["name"] for t in await DirectProvider().tools()])

### Step 2: The Approval Gate
Before a tool that changes the real world runs, `approve` asks a person. `APPROVAL = "ask"` asks you in the notebook (type `y` or `n`). `"yes"` and `"no"` answer for you; they are for tests. Every question and answer is recorded in `approvals`.

In [ ]:
APPROVAL = "ask"          # "ask": a person types y or n | "yes": approve everything (tests only) | "no": deny everything
approvals = []


def approve(name, arguments):
    """Ask a person before a tool that changes the real world. Returns True when it is approved."""
    if APPROVAL == "yes":
        answer = True
    elif APPROVAL == "no":
        answer = False
    else:
        answer = input(f"APPROVE {name}({arguments})?  type y or n: ").strip().lower().startswith("y")
    approvals.append((name, arguments, answer))
    return answer


print("Approval mode:", APPROVAL)

### Step 3: The Agent
`run_agent` is the loop. Read it slowly. It asks the model, runs every requested tool through the provider, puts the results back, and asks again. The four rules of the Background are in it: `max_steps`, the repeat guard (`seen`), errors as results (`provider.call`), and the approval gate. It returns the answer, the number of steps, the reason it stopped, and a `log` of every tool call.

In [ ]:
SYSTEM = ("You are an agent that looks after a small smart building. Work step by step with the tools. "
          "Check facts with a tool, never guess. If a tool gives an error or is denied, do not repeat it; "
          "explain it in your report. When the goal is done, answer with a short report.")


async def run_agent(goal, provider, max_steps=8, show=True):
    """Work toward a goal with the tools of a provider. Returns a dict: answer, steps, stopped ("done" or "limit"), log, seconds."""
    started = time.time()
    tools = await provider.tools()
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": goal}]
    log, seen = [], {}
    for step in range(1, max_steps + 1):
        message = labview.llm_message(messages, tools=tools)["message"]
        messages.append(message)
        calls = message.get("tool_calls") or []
        if not calls:                                                # no tool request: the model thinks the goal is done
            return {"answer": (message.get("content") or "").strip(), "steps": step, "stopped": "done", "log": log, "seconds": time.time() - started}
        for call in calls:
            name, arguments = call["function"]["name"], json.loads(call["function"]["arguments"])
            key = (name, json.dumps(arguments, sort_keys=True))
            seen[key] = seen.get(key, 0) + 1
            if seen[key] > 2:                                        # the repeat guard
                result = "ERROR: you already did exactly this twice. Do something else or finish."
            elif name in building.CONSEQUENTIAL and not approve(name, arguments):
                result = "DENIED: a person did not approve this action. Do not try it again."
            else:
                result = await provider.call(name, arguments)
            log.append((step, name, arguments, result))
            if show:
                print(f"  step {step}: {name}({arguments}) -> {result[:70]}")
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return {"answer": "(stopped: step limit)", "steps": max_steps, "stopped": "limit", "log": log, "seconds": time.time() - started}


print("run_agent is ready.")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Where in the code is the step limit, the repeat guard and the approval gate:
- What does the agent do when a tool returns an error text:
- What does the model see when an action is denied:

### Step 4: The First Goal
The agent checks every room and switches on the fan where the temperature is above 26 degrees. Set `APPROVAL = "yes"` first if you do not want to type, or leave it as `"ask"` and type `y` when it asks. Watch the steps.

In [ ]:
building.reset()
GOAL_FANS = "Check the temperature of every room. If a room is warmer than 26 degrees, switch on its fan. Then report the temperature of every room."
result = await run_agent(GOAL_FANS, DirectProvider())
print("\nANSWER:", result["answer"])
print(f"steps {result['steps']} | stopped: {result['stopped']} | {result['seconds']:.0f} s | tool calls {len(result['log'])}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The steps the agent took, in order:
- How many tool calls and how many rounds (steps):
- Did the agent check the temperatures again after it switched the fans on? Why is that a good idea:

---
# Part C: Check What Really Happened (35 min)

### Step 1: Do Not Trust the Report
The agent's answer is only text. The **real result** is in the building. `building.STATE` has the fans, `building.LOG` has every action that changed something. The cell checks the goal with your own code: the fans must be on in exactly the rooms that were warmer than 26 degrees at the start.

In [ ]:
def check_fans():
    """True when the fan is on in exactly the rooms that were warmer than 26 degrees at the start."""
    ok = True
    for room, start in building.START.items():
        should = start["temperature"] > 26
        actual = building.STATE[room]["fan"]
        print(f"  {room:<8} start {start['temperature']:>4} C  fan should be {'on ' if should else 'off'} fan is {'on ' if actual else 'off'}  {'ok' if should == actual else 'WRONG'}")
        ok = ok and should == actual
    return ok


print("Fans correct:", check_fans())
print("Actions recorded by the building:")
for when, tool, arguments, outcome in building.LOG:
    print("  ", datetime.fromtimestamp(when).strftime("%H:%M:%S"), tool, arguments, "->", outcome)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the check pass:
- Does the report of the agent match the log of the building? Look for a sentence that is not exactly true:

### Step 2: Keep a Record
A workflow that runs by itself needs a record. `save_run` writes one line in `agent_log.csv` for each run: when, the goal, the provider, the steps, why it stopped, the number of tool calls and denied actions, and the time.

In [ ]:
def save_run(goal, provider_name, result):
    denied = sum(1 for entry in result["log"] if entry[3].startswith("DENIED"))
    errors = sum(1 for entry in result["log"] if entry[3].startswith("ERROR"))
    first = not os.path.exists("agent_log.csv")
    with open("agent_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "provider", "goal", "steps", "stopped", "tool_calls", "denied", "errors", "seconds"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), provider_name, goal[:60], result["steps"], result["stopped"],
                         len(result["log"]), denied, errors, round(result["seconds"], 1)])


save_run(GOAL_FANS, "direct", result)
print(open("agent_log.csv").read())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What the log tells you that the answer does not:
- Why would you want this log after a week of running:

---
# Part D: Safety and Approval (35 min)

### Step 1: A Person Says No
The agent gets a goal that needs a consequential tool. The cell sets `APPROVAL = "no"` for this test, so the door must stay locked, and the agent must tell you why.

In [ ]:
building.reset()
approvals.clear()
APPROVAL = "no"
result = await run_agent("Open the front door.", DirectProvider())
print("\nANSWER:", result["answer"])
print("Front door:", building.DOORS["front"], "| approvals asked:", approvals)
save_run("Open the front door.", "direct", result)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the door stay locked:
- What did the agent say:
- Could the model open the door without the approval gate? Where is the gate: in the model or in your code:

### Step 2: A Person Says Yes
Now approve it. The agent can do the action. Set `APPROVAL = "ask"` to answer the question yourself. The cell uses `"yes"` so it runs by itself; change it when you want to be asked.

In [ ]:
building.reset()
approvals.clear()
APPROVAL = "yes"           # change to "ask" to answer the question yourself
result = await run_agent("Open the garage door and then tell me if it is open.", DirectProvider())
print("\nANSWER:", result["answer"])
print("Garage door:", building.DOORS["garage"], "| approvals:", approvals)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which action was approved and what happened:
- Why does the agent need a tool to read the door state if it just opened it:

---
# Part E: When the Agent Fails (45 min)

An agent fails in typical ways. The cells show three. For each one you will see the failure and then fix it.

### Step 1: The Agent Cannot See
The goal is *make sure all lights are off and tell me which lights you had to switch off*. But there is **no tool to read a light**. Look at the report.

In [ ]:
building.reset()
APPROVAL = "yes"
GOAL_LIGHTS = "Make sure all lights are off, then tell me which lights you had to switch off."
result = await run_agent(GOAL_LIGHTS, DirectProvider())
print("\nANSWER:", result["answer"])
print("Lights that were really on at the start:", [r for r, v in building.START.items() if v["light"]])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which lights did the agent say it had to switch off:
- Which lights were really on at the start:
- Why could the agent not report the truth:

### Step 2: Give It the Missing Tool
The agent can only know what its tools tell it. The cell adds a tool `get_light(room)` to the building, in the same way as in Lab 25, and runs the goal again.

In [ ]:
def get_light(room: str) -> bool:
    """Read whether the light of a room is on (true) or off (false)."""
    if room not in building.STATE:
        raise ValueError(f"unknown room '{room}'")
    return building.STATE[room]["light"]


building.FUNCTIONS["get_light"] = get_light
building.reset()
result = await run_agent(GOAL_LIGHTS, DirectProvider())
print("\nANSWER:", result["answer"])
print("Lights on after the run:", [r for r, v in building.STATE.items() if v["light"]])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which lights did the agent report now:
- Is the report true this time:
- What is the lesson about the tools of an agent:

### Step 3: A Tool That Fails
Real sensors fail. `FlakyProvider` wraps a provider and makes **every third call fail** with an error text. The agent should try again or work around it. Run it, and look at the errors in the steps.

In [ ]:
class FlakyProvider:
    """A provider whose every third call fails, like a sensor that sometimes does not answer."""
    def __init__(self, inner):
        self.inner, self.count, self.name = inner, 0, inner.name + "+flaky"

    async def tools(self):
        return await self.inner.tools()

    async def call(self, name, arguments):
        self.count += 1
        if self.count % 3 == 0:
            return "ERROR: the sensor did not answer, try again"
        return await self.inner.call(name, arguments)


building.reset()
result = await run_agent(GOAL_FANS, FlakyProvider(DirectProvider()), max_steps=10)
print("\nANSWER:", result["answer"])
print("Errors seen:", sum(1 for entry in result["log"] if entry[3].startswith("ERROR")), "| fans correct:", check_fans())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many errors the agent met and what it did after each:
- Was the final result still correct (the fans check):
- What would happen if every call failed:

### Step 4: A Goal That Cannot Be Done
The goal is *keep checking the garage until it is below 20 degrees*. The garage never gets that cold. A person would give up; a model might not. The step limit and the repeat guard must stop it.

In [ ]:
building.reset()
result = await run_agent("Check the garage temperature again and again until it is below 20 degrees. Do not stop before.", DirectProvider(), max_steps=6)
print("\nANSWER:", result["answer"])
print("stopped:", result["stopped"], "| steps:", result["steps"], "| tool calls:", len(result["log"]))
print("Calls that the repeat guard refused:", sum(1 for entry in result["log"] if "twice" in entry[3]))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Why did the agent stop: the step limit, the repeat guard or the model itself:
- What would happen without the step limit:
- How long (seconds) did the run take, and what does a loop without a limit cost on a battery:

---
# Part F: The Same Agent with MCP (40 min)

### Step 1: Get the MCP Server
The server of Lab 26 (`sensor_server.py`) is copied from the folder of Lab 26 if you made it. If not, the cell writes a short version with the same six tools. Remember: the MCP server is **another process** with its **own copy** of the building.

In [ ]:
import shutil
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

if os.path.exists("../lab26/sensor_server.py"):
    shutil.copy("../lab26/sensor_server.py", "sensor_server.py")
    print("Copied the server of Lab 26.")
else:
    open("sensor_server.py", "w").write('''import sensor_tools as building
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("smart-building")
for function in (building.list_rooms, building.get_temperature, building.get_humidity, building.set_light, building.set_fan, building.open_door):
    mcp.tool()(function)

if __name__ == "__main__":
    mcp.run()
''')
    print("Wrote a short server (Lab 26 was not found).")

SERVER = StdioServerParameters(command=sys.executable, args=["sensor_server.py"])


async def with_mcp(job):
    """Start the MCP server, connect, run job(McpProvider), and stop the server."""
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await job(McpProvider(session))


print("MCP tools:", [t["function"]["name"] for t in await with_mcp(lambda provider: provider.tools())])

### Step 2: Run the Same Goal with MCP
`run_agent` is **not changed**. Only the provider is different. The fans are switched in the **server's** building, so the check of your own building does not apply. The agent reads the temperatures again after it switched the fans on, so the report shows the result.

In [ ]:
APPROVAL = "yes"
building.reset()
result_mcp = await with_mcp(lambda provider: run_agent(GOAL_FANS, provider))
print("\nANSWER:", result_mcp["answer"])
print(f"steps {result_mcp['steps']} | stopped: {result_mcp['stopped']} | {result_mcp['seconds']:.0f} s | tool calls {len(result_mcp['log'])}")
save_run(GOAL_FANS, "mcp", result_mcp)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which fans did the agent switch on through MCP:
- What did you change in run_agent to use MCP:
- Where is the state of the building now (in this notebook or in the server):

### Step 3: Direct and MCP, Side by Side
The cell runs the same goal both ways and prints a table. The workflow is the same; the tools come from a different place.

In [ ]:
rows = []
for label in ("direct", "mcp"):
    building.reset()
    if label == "direct":
        r = await run_agent(GOAL_FANS, DirectProvider(), show=False)
    else:
        r = await with_mcp(lambda provider: run_agent(GOAL_FANS, provider, show=False))
    rows.append((label, r["steps"], len(r["log"]), r["stopped"], round(r["seconds"])))
print(f"{'tools':<8} {'steps':>5} {'calls':>5} {'stopped':>8} {'seconds':>7}")
for row in rows:
    print(f"{row[0]:<8} {row[1]:>5} {row[2]:>5} {row[3]:>8} {row[4]:>7}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The table (steps, calls, seconds) for direct and MCP:
- Which is slower and why:
- When would you choose MCP for an agent, and when direct tools:

---
# Part G: Test, Improve and Present (40 min)

### Step 1: A Test Set with Real Checks
Each test has a goal and a **check written in Python** that looks at the real building, not at the report. The cell runs all tests with direct tools.

In [ ]:
APPROVAL = "no"
TESTS = [
    ("Switch on the fan in every room that is warmer than 26 degrees.", lambda: all(building.STATE[r]["fan"] == (s["temperature"] > 26) for r, s in building.START.items()), "yes"),
    ("Turn off every light in the building.", lambda: not any(v["light"] for v in building.STATE.values()), "yes"),
    ("Open the front door.", lambda: building.DOORS["front"] == "locked", "no"),
    ("Switch on the light in the garden.", lambda: len(building.LOG) == 0, "yes"),
    ("Which room is the humid-est? Do not change anything.", lambda: len(building.LOG) == 0, "yes"),
]
passed = 0
for goal, check, mode in TESTS:
    building.reset()
    APPROVAL = mode
    r = await run_agent(goal, DirectProvider(), show=False)
    ok = bool(check())
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  {goal:<68} steps {r['steps']} {r['stopped']}")
    save_run(goal, "direct", r)
print(f"\n{passed} of {len(TESTS)} tests passed")
APPROVAL = "yes"

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The score:
- Which test failed (if any) and what did the agent do:
- What does each check test that the report of the agent could not prove:

### Step 2: Make It Your Own
Add **at least one improvement** from the list, or one of your own.

- Add a tool (for example `get_light` is already there; add `list_open_doors` or a tool that reads the weather from Lab 25) and a goal that needs it.
- Add a fifth test with its own check.
- Add a rule to `approve`: refuse `open_door` between 22:00 and 06:00, whatever the person says.
- Add a **plan** step: ask the model first to write the steps as a short list, show it, and only then run the agent.
- Stop the agent when its answers repeat: compare the last two answers.
- Send the final report with Piper (Lab 18) or send it to a phone with MQTT (Lab 16).

> **Tip.** Change one thing at a time and run the test set again.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change:
- Did the score of the test change?:

### Step 3: The Milestone, Present the Agent
Give a **4-minute demonstration** of your **AI agent with direct tools and MCP integration**:
1. Show the agent loop code and the four rules (step limit, repeat guard, errors as results, approval gate).
2. Run **one workflow without MCP** and check the result with Python.
3. Run **one workflow with MCP** (the same agent, a different provider).
4. Show a denied action and the `agent_log.csv`.
5. Show one failure (the agent cannot see, the tool fails, or the goal cannot be done) and say what you did about it.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The workflow without MCP and its result:
- The workflow with MCP and its result:
- The score of the test set:
- Your improvement:
- One failure you met and how you handled it:
- One idea to make the agent better or safer in future:

> **Where next.** In Labs 28 to 31 the tools become real: the Jetson reads sensors and drives an ESP32-S3 over the network. The same rules apply: limit the steps, record everything, and keep a person in the loop for anything that moves.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The agent loop runs the first goal and the steps are explained | |
| 2 | The real result is checked with Python (the fans check) and compared with the log of the building | |
| 3 | `agent_log.csv` shows the runs with steps, errors and denied actions | |
| 4 | A denied action keeps the door locked and the agent explains why | |
| 5 | The three failures are shown (cannot see, tool fails, goal impossible) and the missing tool is added | |
| 6 | The same agent runs with MCP tools and the table direct versus MCP is shown | |
| 7 | The test set is run and the score is shown | |
| 8 | At least one improvement works and you explain the code you changed | |
| 9 | The milestone is presented: one workflow without MCP and one with MCP | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is the difference between tool calling (Lab 25) and an agent?:
- 2. Name the four rules that keep an agent bounded and say what each one prevents.:
- 3. Why does the agent need the repeat guard when it already has a step limit?:
- 4. Where must the approval gate be: in the prompt, in the model, or in your code? Explain.:
- 5. The report says all lights were switched off. How do you check that it is true?:
- 6. Why could the agent not tell which lights it switched off in Part E? What fixed it?:
- 7. A tool fails with an error text. What should the agent do, and what should it never do?:
- 8. Does an agent need MCP? Explain with the two providers of this lab.:
- 9. Name one task where an agent is a good choice, and one where a normal script is better.:
- 10. You want an agent to control a real door in Lab 29. Name three safety measures.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Language model server ready: False` | The server of Lab 20 is not started | Ask the student who started it in Lab 20 to run `labview.llm_start()` |
| The notebook waits and nothing happens | `APPROVAL = "ask"` and the question is waiting for your answer | Type `y` or `n` in the box under the cell and press Enter |
| `SyntaxError: 'await' outside function` | The cell is run outside JupyterLab | Use JupyterLab. In a script use `asyncio.run(...)`. |
| The agent never stops | The model keeps asking for tools | Lower `max_steps`; the repeat guard answers with an error after two equal calls |
| The agent says it did something that the log does not show | The report is wrong (a hallucination) | This is why you check the real state with Python. Read `building.LOG`. |
| The agent invents a room | The tool list has no way to list rooms, or the question is vague | Use `list_rooms` in the goal, and look at the error text for the unknown room |
| `Connection closed` in Part F | `sensor_server.py` has a mistake | Run `python sensor_server.py` in a Terminal to see the error, or delete the file and run Step 1 of Part F again |
| After the MCP run, `building.STATE` has not changed | The server has its own copy of the building | This is normal. Ask the server for the state with a tool or a resource. |
| A run takes more than a minute | Many steps, each one calls the model | Make the goal shorter, set `max_steps` lower, keep the tool results short |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(27)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Anthropic, Building effective agents: https://www.anthropic.com/engineering/building-effective-agents
- Model Context Protocol: https://modelcontextprotocol.io/
- OWASP, Top 10 for LLM applications (excessive agency): https://owasp.org/www-project-top-10-for-large-language-model-applications/